# BenchPress: LLM Evaluation

How do you actually measure if a language model is good? Run standard benchmarks, build a custom one, and learn why evaluation is harder than it looks.

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from run_eval import run_benchmark, extract_scores, run_comparison, print_leaderboard
from custom_bench import evaluate_custom, SPANISH_BENCH
from prompt_sensitivity import test_sensitivity, print_sensitivity_report, PROMPT_TEMPLATES
from model_info import print_model_table

## 0. Model overview

What are we comparing? Let's look at model sizes first.

In [ ]:
print_model_table()

## 1. Standard benchmarks on small models

We use lm-eval-harness to run HellaSwag and ARC on GPT-2 sizes. These are commonsense reasoning tasks.

In [ ]:
# run comparison on small models
models = ["gpt2", "gpt2-medium", "EleutherAI/pythia-160m", "EleutherAI/pythia-410m"]

scores = run_comparison(
    models=models,
    tasks="quick",  # hellaswag + arc_easy
    limit=200,
    num_fewshot=0,
)

print_leaderboard(scores)

In [ ]:
# plot the leaderboard
model_names = list(scores.keys())
task_names = set()
for s in scores.values():
    task_names.update(k for k in s.keys() if k != "error")
task_names = sorted(task_names)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(model_names))
width = 0.8 / max(len(task_names), 1)

colors = ["#3498db", "#2ecc71", "#f39c12", "#e74c3c"]
for i, task in enumerate(task_names):
    vals = [scores[m].get(task, 0) for m in model_names]
    label = task.split("/")[-1]
    ax.bar(x + i * width, vals, width, label=label, color=colors[i % len(colors)])

ax.set_xticks(x + width * (len(task_names) - 1) / 2)
ax.set_xticklabels([m.split("/")[-1] for m in model_names], rotation=15)
ax.set_ylabel("Accuracy")
ax.set_title("Model comparison (0-shot)")
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
os.makedirs("plots", exist_ok=True)
plt.savefig("plots/leaderboard.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Few-shot matters

Same model, same task, different number of examples in the prompt. How much does it change?

In [ ]:
fewshot_results = {}
for n in [0, 1, 3, 5]:
    results = run_benchmark("gpt2-medium", ["hellaswag"], limit=200, num_fewshot=n)
    fewshot_results[n] = extract_scores(results)
    print(f"  {n}-shot: {fewshot_results[n]}")

fig, ax = plt.subplots(figsize=(8, 5))
shots = list(fewshot_results.keys())
# get first metric value from each
accs = [list(fewshot_results[n].values())[0] if fewshot_results[n] else 0 for n in shots]
ax.bar(range(len(shots)), accs, color="#3498db", tick_label=[f"{n}-shot" for n in shots])
ax.set_ylabel("Accuracy")
ax.set_title("GPT-2 Medium on HellaSwag: few-shot effect")
for i, v in enumerate(accs):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.savefig("plots/fewshot.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Prompt sensitivity

Same questions, different prompt format. The model gives different answers.

In [ ]:
sensitivity_results = test_sensitivity("gpt2")
print_sensitivity_report(sensitivity_results)

# visualize
templates = list(PROMPT_TEMPLATES.keys())
accs = []
for t in templates:
    correct = sum(1 for r in sensitivity_results if r.get(f"{t}_correct", False))
    accs.append(correct / len(sensitivity_results))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#3498db", "#2ecc71", "#f39c12", "#e74c3c", "#9b59b6"]
ax.bar(templates, accs, color=colors[:len(templates)])
ax.set_ylabel("Accuracy")
ax.set_title("GPT-2: same questions, different prompt formats")
ax.set_ylim(0, 1)
for i, v in enumerate(accs):
    ax.text(i, v + 0.02, f"{v:.0%}", ha="center")

plt.tight_layout()
plt.savefig("plots/prompt_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Custom Spanish benchmark

In [ ]:
print(f"Custom benchmark: {len(SPANISH_BENCH)} questions\n")
for i, q in enumerate(SPANISH_BENCH[:5]):
    print(f"{i+1}. {q['question']}")
    for c in q['choices']:
        marker = ' *' if c[0] == q['answer'] else ''
        print(f"   {c}{marker}")
    print()

In [ ]:
# test different models on the Spanish benchmark
spanish_results = []
for model_name in ["gpt2", "gpt2-medium"]:
    result = evaluate_custom(model_name=model_name, template="plain")
    spanish_results.append(result)

# also test with English template
result_en = evaluate_custom(model_name="gpt2", template="english")
spanish_results.append({**result_en, "model": "gpt2 (english template)"})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
names = [r["model"] for r in spanish_results]
accs = [r["accuracy"] for r in spanish_results]
ax.bar(range(len(names)), accs, color=["#3498db", "#2ecc71", "#f39c12"])
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.split("/")[-1] for n in names], rotation=15)
ax.set_ylabel("Accuracy")
ax.set_title("Custom Spanish benchmark")
ax.set_ylim(0, 1)
ax.axhline(y=0.25, color="gray", linestyle="--", alpha=0.5, label="Random chance (25%)")
ax.legend()
for i, v in enumerate(accs):
    ax.text(i, v + 0.02, f"{v:.0%}", ha="center")

plt.tight_layout()
plt.savefig("plots/spanish_bench.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Template comparison on Spanish bench

Does it help to ask in English even if the questions are in Spanish?

In [ ]:
from compare_templates import compare_all_templates

template_scores = compare_all_templates("gpt2")

fig, ax = plt.subplots(figsize=(8, 5))
names = list(template_scores.keys())
vals = list(template_scores.values())
ax.bar(names, vals, color=["#3498db", "#2ecc71", "#f39c12"])
ax.set_ylabel("Accuracy")
ax.set_title("Spanish bench: template language matters")
ax.set_ylim(0, 1)
ax.axhline(y=0.25, color="gray", linestyle="--", alpha=0.5)
for i, v in enumerate(vals):
    ax.text(i, v + 0.02, f"{v:.0%}", ha="center")
plt.tight_layout()
plt.savefig("plots/template_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Key takeaways

**Prompt format changes scores significantly.**

The same model on the same questions can score 20-40% differently just by changing how you format the prompt. This is why comparing numbers across different eval setups is misleading.

**Few-shot helps, but not always.**

Adding examples to the prompt usually improves scores, but the gain varies by task. Some tasks barely change, others jump a lot.

**Small models struggle with non-English.**

GPT-2 was trained mostly on English text. On Spanish questions, it performs near random chance. Bigger models or multilingual models do better.

**Benchmarks are not the full picture.**

A model can score well on MMLU but be terrible at following instructions. Benchmarks measure narrow capabilities. Real usefulness is harder to quantify.